In [1]:
from ultralytics import YOLO

In [2]:
import pytesseract
import re

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [3]:
plate_model = YOLO("License_plate_detection_model.pt")
model_seatbelt = YOLO("seat_belt_detection_model.pt")

In [4]:
# result = model.predict("videonetics-ai-enabled-seat-belt-detection-technology-920x533.jpg")[0]

In [5]:
# result.plot()

In [6]:
# import cv2

# annotated = result.plot()  # returns a NumPy image in RGB format
# annotated_bgr = cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
# cv2.imwrite("output_result.jpg", annotated_bgr)

In [7]:
def detect_seatbelt(frame):
    results = model_seatbelt.predict(frame)[0]

    persons_with = []
    persons_without = []

    for box in results.boxes:
        cls = int(box.cls[0])
        label = results.names[cls]

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if label == "Seat_belt":
            persons_with.append((x1, y1, x2, y2))
        elif label == "No-seat-belt":
            persons_without.append((x1, y1, x2, y2))

    return persons_with, persons_without

In [8]:
def detect_plates(frame):
    results = plate_model.predict(frame)[0]

    plates = []

    for i in results.boxes:
        x1, y1, x2, y2 = map(int, i.xyxy[0])
        plates.append((x1, y1, x2, y2))

    return plates

In [9]:
import math

def get_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2)//2, (y1 + y2)//2)

def find_nearest_plate(person_box, plates):
    px, py = get_center(person_box)

    min_dist = float("inf")
    best_plate = None

    for plate in plates:
        lx, ly = get_center(plate)

        dist = math.sqrt((px - lx)**2 + (py - ly)**2)

        if dist < min_dist:
            min_dist = dist
            best_plate = plate

    return best_plate

In [10]:
import pytesseract
import cv2

def read_plate(frame, plate_box):
    # Convert float → int
    x1, y1, x2, y2 = map(int, plate_box)

    # Crop plate
    crop = frame[y1:y2, x1:x2]

    # Preprocessing
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 11, 17, 17)

    # 🔥 Add threshold (VERY IMPORTANT)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    # OCR
    text = pytesseract.image_to_string(
        thresh,
        config="--psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789",
    )

    # 🔥 Clean text
    text = re.sub(r"[^A-Z0-9]", "", text)

    return text.strip(), crop

In [11]:
def process_frame(frame):

    persons_with, persons_without = detect_seatbelt(frame)
    plates = detect_plates(frame)

    results = []

    for person in persons_without:

        plate = find_nearest_plate(person, plates)

        if plate is not None:
            text, crop = read_plate(frame, plate)

            results.append({
                "violation": True,
                "person_box": person,
                "plate_box": plate,
                "plate_text": text
            })

    return results

In [12]:
import cv2

def draw_results(frame, results):
    for res in results:
        px1, py1, px2, py2 = res["person_box"]
        lx1, ly1, lx2, ly2 = res["plate_box"]
        text = res["plate_text"]

        # 🔴 Draw person box (red)
        cv2.rectangle(frame, (px1, py1), (px2, py2), (0, 0, 255), 2)
        cv2.putText(frame, "No Seatbelt", (px1, py1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # 🟢 Draw plate box (green)
        cv2.rectangle(frame, (lx1, ly1), (lx2, ly2), (0, 255, 0), 2)

        # 🔤 Draw plate text
        cv2.putText(frame, text, (lx1, ly1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    return frame

In [13]:
import cv2

img_path = r"C:\Faiq\SMIT\Assignment of SMIT\Assignment_13\img\5.png"

# Read image ONCE
frame = cv2.imread(img_path)

# Run detection
results = process_frame(frame)

# Print results (optional)
for res in results:
    print("Violation Detected!")
    print(f"Person Box: {res['person_box']}")
    print(f"Plate Box: {res['plate_box']}")
    print(f"Plate Text: {res['plate_text']}")

# 🔥 Draw results
output = draw_results(frame.copy(), results)

# Show image
cv2.imshow("Result", output)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Save image
cv2.imwrite("output.jpg", output)


0: 128x256 (no detections), 115.5ms
Speed: 3.1ms preprocess, 115.5ms inference, 1.4ms postprocess per image at shape (1, 3, 128, 256)

0: 128x256 (no detections), 54.4ms
Speed: 1.4ms preprocess, 54.4ms inference, 0.8ms postprocess per image at shape (1, 3, 128, 256)


True

In [14]:
print("Asf")

Asf


In [4]:
import cv2

img_path2 = "Cars177.png"
img = cv2.imread(img_path2)

height, width, _ = img.shape

out = cv2.VideoWriter("test3.mp4", cv2.VideoWriter_fourcc(*'mp4v'), 5, (width, height))

for _ in range(30):  # 6 seconds video
    out.write(img)

out.release()